In [1]:
!pip install earthengine-api geemap joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 47.7 MB/s eta 0:00:00


In [2]:
import ee
import geemap
import numpy as np
import joblib
from datetime import datetime

In [3]:
ee.Authenticate()
ee.Initialize(project='gee-jammu-dissertation')

In [5]:
model = joblib.load("Best_model.pkl")

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator HalvingGridSearchCV from version 1.7.1 when using version 1.6.1. This might lead t

In [39]:
lat = 32.7266
lon = 74.8570

point = ee.Geometry.Point([lon, lat])

In [40]:
today = datetime.today()

year = today.year
month = today.month

print(f"Prediction for: {month}/{year}")

Prediction for: 5/2026


In [41]:
# Date range (last 3 months)
start_date = f"{year}-01-01"
end_date   = f"{year}-03-31"

In [42]:
def select_bands(img):
    return img.select(['B2','B3','B4','B8','B11'])

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(point)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .map(select_bands)
    .median()
)

ndvi = s2.normalizedDifference(['B8','B4']).rename('NDVI')
evi = s2.expression(
    '2.5*((NIR - RED)/(NIR + 6*RED - 7.5*BLUE + 1))',
    {
        'NIR': s2.select('B8'),
        'RED': s2.select('B4'),
        'BLUE': s2.select('B2')
    }
).rename('EVI')

ndmi = s2.normalizedDifference(['B8','B11']).rename('NDMI')
ndwi = s2.normalizedDifference(['B3','B8']).rename('NDWI')

In [43]:
lst = (
    ee.ImageCollection("MODIS/061/MOD11A2")
    .filterBounds(point)
    .filterDate(start_date, end_date)
    .select('LST_Day_1km')
    .mean()
    .multiply(0.02)
    .subtract(273.15)
    .rename('LST')
)

In [44]:
rain = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterBounds(point)
    .filterDate(start_date, end_date)
    .sum()
    .rename('Rainfall')
)

rain_lag = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterBounds(point)
    .filterDate(f"{year-1}-10-01", f"{year-1}-12-31")
    .sum()
    .rename('Rainfall_lag1')
)

In [45]:
feature_image = ee.Image.cat([
    ndvi, evi, ndmi, ndwi,
    lst, rain, rain_lag
])

values = feature_image.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=point,
    scale=30
).getInfo()

print(values)

{'EVI': 0.4298203011170471, 'LST': 19.324545454545444, 'NDMI': -0.017721518987341773, 'NDVI': 0.043265705206550965, 'NDWI': -0.07832238504295098, 'Rainfall': 164.59881973266602, 'Rainfall_lag1': 79.26630678772926}


In [46]:
features = [
    'NDVI','EVI','NDMI','NDWI',
    'LST','Rainfall','Rainfall_lag1'
]

x = np.array([values[f] for f in features], dtype='float32')

In [47]:
X_model = np.repeat(x.reshape(1,-1), 6, axis=1)

In [48]:
prediction = model.predict(X_model)[0]

In [49]:
print("📍 Location:", lat, lon)
print(f"📅 Month-Year: {month}-{year}")
print("🌾 Predicted Crop Health Score (CHS):", round(prediction, 4))

📍 Location: 32.7266 74.857
📅 Month-Year: 5-2026
🌾 Predicted Crop Health Score (CHS): 0.4932


In [50]:
def classify_chs(chs):
    if chs < 0.30:
        return "LOW"
    elif chs < 0.60:
        return "MEDIUM"
    else:
        return "HIGH"

In [30]:
def get_recommendation(chs_class):
    if chs_class == "LOW":
        return [
            "Increase irrigation",
            "Check soil moisture",
            "Inspect for pests/disease",
            "Apply nutrients if needed"
        ]

    elif chs_class == "MEDIUM":
        return [
            "Monitor crop regularly",
            "Optimize irrigation schedule",
            "Maintain nutrient balance"
        ]

    else:  # HIGH
        return [
            "Maintain current practices",
            "Avoid over-irrigation",
            "Continue periodic monitoring"
        ]

In [31]:
chs_value = prediction

chs_class = classify_chs(chs_value)

recommendations = get_recommendation(chs_class)

In [32]:
print("\n📍 Location:", lat, lon)
print(f"📅 Month-Year: {month}-{year}")
print("🌾 CHS:", round(chs_value, 4))
print("📊 Category:", chs_class)

print("\n📌 Recommended Actions:")
for r in recommendations:
    print("•", r)


📍 Location: 33.848806 74.88575
📅 Month-Year: 5-2026
🌾 CHS: 0.5109
📊 Category: MEDIUM

📌 Recommended Actions:
• Monitor crop regularly
• Optimize irrigation schedule
• Maintain nutrient balance


In [33]:
results = [
    {
        "name": "Location 1",
        "lat": 33.848806,
        "lon": 74.885750,
        "chs": 0.67
    },
    {
        "name": "Location 2",
        "lat": 32.663750,
        "lon": 74.816806,
        "chs": 0.45
    }
]

In [34]:
def classify_chs(chs):
    if chs < 0.30:
        return "LOW"
    elif chs < 0.60:
        return "MEDIUM"
    else:
        return "HIGH"

def get_color(chs_class):
    if chs_class == "LOW":
        return "red"
    elif chs_class == "MEDIUM":
        return "orange"
    else:
        return "green"

In [35]:
import folium

m = folium.Map(
    location=[33.2, 75.0],
    zoom_start=8,
    tiles="OpenStreetMap",
    control_scale=True
)

In [36]:
for r in results:
    chs_class = classify_chs(r["chs"])
    color = get_color(chs_class)

    popup_text = f"""
    <b>{r['name']}</b><br>
    CHS: {round(r['chs'], 3)}<br>
    Category: {chs_class}
    """

    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=8,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(popup_text, max_width=250)
    ).add_to(m)

In [37]:
legend_html = '''
<div style="
position: fixed;
bottom: 40px; left: 40px; width: 180px;
background-color: white; z-index:9999;
border:2px solid grey; padding: 10px;
font-size:14px;
">
<b>CHS Classification</b><br><br>

<div><span style="color:red;">●</span> Low (Stressed)</div>
<div><span style="color:orange;">●</span> Medium</div>
<div><span style="color:green;">●</span> High</div>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

In [38]:
m.save("CHS_points_map.html")